In [67]:
# Imports
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import utils

In [68]:
def plot_all_data(df_list, filenames, save_path, out_name):
    """
    Plot all L-ending files into one figure, all M-ending files into another.
    Each figure has N subplots (N = number of features excluding 'Time'/'label' if present).
    """
    if len(df_list) != len(filenames):
        raise ValueError(f"Length mismatch: len(df_list)={len(df_list)} vs len(filenames)={len(filenames)}")

    os.makedirs(save_path, exist_ok=True)

    # Map filename -> data
    file2df = {fn: df for fn, df in zip(filenames, df_list)}

    def to_dataframe(x):
        # Accept pandas DataFrame or numpy 2D array
        if isinstance(x, pd.DataFrame):
            return x
        x = np.asarray(x)
        if x.ndim != 2:
            raise ValueError(f"Expected 2D array for plotting, got shape={x.shape}")
        return pd.DataFrame(x)

    def group_suffix(fn):
        # Use stem (without extension) to check ending char
        stem = Path(fn).stem
        if stem.endswith("L"):
            return "L"
        if stem.endswith("M"):
            return "M"
        return None

    groups = {"L": [], "M": []}
    for fn in filenames:
        suf = group_suffix(fn)
        if suf in groups:
            groups[suf].append(fn)

    for suf in ["L", "M"]:
        group_files = groups[suf]
        if not group_files:
            continue

        group_dfs = [to_dataframe(file2df[fn]) for fn in group_files]
        max_len = max(len(df) for df in group_dfs)

        # Feature selection (exclude Time/label if present)
        cols = list(group_dfs[0].columns)
        features = [c for c in cols if str(c) not in ["Time", "label"]]

        # If features empty (e.g., numeric columns but all got filtered), fallback to all columns
        if len(features) == 0:
            features = cols

        n_feat = len(features)
        fig_h = max(2.2 * n_feat, 3.0)
        fig, axes = plt.subplots(n_feat, 1, figsize=(12, fig_h), sharex=True)
        if n_feat == 1:
            axes = [axes]

        for df, fn in zip(group_dfs, group_files):
            x = np.arange(len(df))
            label = Path(fn).stem
            for i, feat in enumerate(features):
                # Skip missing columns gracefully
                if feat not in df.columns:
                    continue
                axes[i].plot(x, df[feat].to_numpy(), label=label, linewidth=1.0)

        for i, feat in enumerate(features):
            axes[i].set_ylabel(str(feat))
            axes[i].grid(True, alpha=0.25)

        axes[-1].set_xlim(0, max_len - 1)
        axes[-1].set_xlabel("Index")

        # Put legend on the first subplot (avoid repeating)
        axes[0].legend(ncol=2, fontsize=8, frameon=False)

        fig.suptitle(f"{out_name}{suf}", y=0.995)
        fig.tight_layout()

        out_file = os.path.join(save_path, f"{out_name}{suf}.png")
        fig.savefig(out_file, dpi=200, bbox_inches="tight")
        plt.close(fig)

In [69]:
# Load original .mat files -> df_list, then plot
dataset_folder = '../data/GPVS-Faults'
processed_data_folder = '../data/processed'
save_path = '../data/plot_all'

window_nb = 35

FEATURE_NAMES = [
    "Ipv","Vpv","Vdc",
    "ia","ib","ic",
    "va","vb","vc",
    "Iabc","If",
    "Vabc","Vf"
]

out_name_prefix = "processed_"

In [70]:
filenames = sorted([f for f in os.listdir(dataset_folder) if f.endswith('.mat')])
print(f"Found data files: {filenames}")

df_list = []
for fn in filenames:
    label_str = fn[:3]
    path = os.path.join(dataset_folder, fn)
    df = utils.read_mat_file(path, label=label_str)
    df_list.append(df)

plot_all_data(df_list, filenames, save_path, out_name="orig_")

Found data files: ['F0L.mat', 'F0M.mat', 'F1L.mat', 'F1M.mat', 'F2L.mat', 'F2M.mat', 'F3L.mat', 'F3M.mat', 'F4L.mat', 'F4M.mat', 'F5L.mat', 'F5M.mat', 'F6L.mat', 'F6M.mat', 'F7L.mat', 'F7M.mat']


In [71]:
# Load processed .npy files -> df_processed_list (use arr[window_nb]) -> plot
npy_files = sorted([f for f in os.listdir(processed_data_folder) if f.endswith(".npy")])

name2file = {}
for fn in npy_files:
    stem = Path(fn).stem

    # Base (F0)
    if stem.startswith("X_test_LPPT"):
        name2file["F0L"] = fn
        continue
    if stem.startswith("X_test_MPPT"):
        name2file["F0M"] = fn
        continue

    # Faults (F1..F7)
    m = re.match(r"^X_test_F(\d+)([LM])_", stem)
    if m:
        fx = m.group(1)
        lm = m.group(2)
        name2file[f"F{fx}{lm}"] = fn

def sort_key(name: str):
    m = re.match(r"^F(\d+)([LM])$", name)
    if not m:
        return (99, 99)
    fidx = int(m.group(1))
    lm = m.group(2)
    return (0 if lm == "L" else 1, fidx)

processed_filenames = sorted(name2file.keys(), key=sort_key)

df_processed_list = []
for short_name in processed_filenames:
    real_fn = name2file[short_name]
    path = os.path.join(processed_data_folder, real_fn)

    arr = np.load(path)  # expected shape: (xxx, 200, 13)
    if arr.ndim != 3:
        raise ValueError(f"{real_fn}: expected 3D array, got shape={arr.shape}")
    if window_nb < 0 or window_nb >= arr.shape[0]:
        raise IndexError(f"{real_fn}: window_nb={window_nb} out of range for shape={arr.shape}")

    window = arr[window_nb]  # shape: (200, 13)

    if window.shape[1] != len(FEATURE_NAMES):
        raise ValueError(
            f"{real_fn}: feature number mismatch, got {window.shape[1]}, expected {len(FEATURE_NAMES)}"
        )

    df = pd.DataFrame(window, columns=FEATURE_NAMES)
    df_processed_list.append(df)

plot_all_data(df_processed_list, processed_filenames, save_path, out_name=f"{out_name_prefix}{window_nb}_")